# 02 - Baseline Evaluation Notebook

This notebook evaluates the ratio-based similarity baseline model.

## Contents

1. **Load Data**: Load player profiles and events
2. **Fit Baseline**: Train the ratio-based similarity model
3. **Evaluate Metrics**: Recall@K, NDCG, embedding diagnostics
4. **Similarity Examples**: Show example queries
5. **Visualization**: Embedding space, similarity heatmaps

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.decomposition import PCA
import torch

import sys
sys.path.append('..')

from baselines import RatioSimilarityBaseline, RatioSimilarityConfig, RoleDiscovery
from training.metrics import compute_recall_at_k, compute_embedding_statistics

print("Imports loaded successfully!")

## 1. Load/Create Player Data

In production, load from preprocessed data files. Here we create synthetic data for demonstration.

In [ ]:
def create_synthetic_profiles(n_players=100):
    """Create synthetic player profiles for testing."""
    np.random.seed(42)
    
    profiles = {}
    for i in range(n_players):
        pid = f"player_{i}"
        
        # Random spatial heatmap
        heatmap = np.random.exponential(0.1, (8, 12))
        heatmap = heatmap / heatmap.sum()
        
        profiles[pid] = {
            'spatial': {
                'all': heatmap,
                'pass': heatmap * np.random.uniform(0.8, 1.2, heatmap.shape),
                'receive': heatmap * np.random.uniform(0.8, 1.2, heatmap.shape),
                'pressure': heatmap * np.random.uniform(0.8, 1.2, heatmap.shape),
            },
            'chains': np.random.rand(20),
            'value_added': {
                'xT_gained': np.random.uniform(0, 0.5),
                'dangerous_pass_share': np.random.uniform(0, 0.3),
                'progressive_share': np.random.uniform(0, 0.4),
            },
            'pressing': {
                'pressure_rate': np.random.uniform(0, 0.2),
                'high_press_rate': np.random.uniform(0, 0.3),
            },
            'passing': {
                'forward_pass_rate': np.random.uniform(0.3, 0.7),
                'progressive_pass_rate': np.random.uniform(0.1, 0.4),
                'pass_length_mean': np.random.uniform(10, 30),
                'verticality': np.random.uniform(-0.5, 0.5),
            },
            'receiving': {
                'receive_rate': np.random.uniform(0.1, 0.3),
                'receive_final_third': np.random.uniform(0, 0.5),
                'receive_central': np.random.uniform(0.3, 0.7),
            },
            'transition': {
                'transition_involvement': np.random.uniform(0, 0.2),
                'transition_directness': np.random.uniform(0, 1),
            },
        }
    
    return profiles

# Create synthetic data
print("Creating synthetic player profiles...")
profiles = create_synthetic_profiles(100)
print(f"✓ Created {len(profiles)} player profiles")

## 2. Fit Baseline Model

In [ ]:
print("Fitting baseline model...")

# Initialize model
config = RatioSimilarityConfig(
    embedding_dim=32,
    n_roles=10,
)
baseline = RatioSimilarityBaseline(config)

# Fit
baseline.fit(profiles)

# Discover roles
print("\nDiscovering roles...")
role_discovery = RoleDiscovery(config)
roles = role_discovery.fit(profiles)
baseline.player_roles = roles

print(f"\nDiscovered roles:")
role_counts = defaultdict(int)
for pid, role_info in roles.items():
    role_counts[role_info['primary_role']] += 1

for role, count in sorted(role_counts.items(), key=lambda x: -x[1]):
    print(f"  {role}: {count} players")

## 3. Evaluate Retrieval Metrics

In [ ]:
print("Evaluating retrieval metrics...")

# Get embeddings as tensors
player_ids = baseline.player_ids
embeddings = torch.tensor(baseline.embeddings).float()

# Create labels (each player is their own class)
labels = torch.arange(len(player_ids))

# Compute metrics
recall_metrics = compute_recall_at_k(embeddings, labels, k_values=[1, 5, 10])
print("\nRecall@K (self-retrieval baseline):")
for k, v in recall_metrics.items():
    print(f"  {k}: {v:.3f}")

# Embedding statistics
stats = compute_embedding_statistics(embeddings)
print("\nEmbedding Statistics:")
print(f"  Norm mean: {stats['norm_mean']:.3f} (std: {stats['norm_std']:.3f})")
print(f"  Similarity mean: {stats['sim_mean']:.3f} (std: {stats['sim_std']:.3f})")
print(f"  Dead dimensions: {stats['n_dead_dims']}")

## 4. Similarity Query Examples

In [ ]:
print("=" * 60)
print("SIMILARITY QUERY EXAMPLES")
print("=" * 60)

# Query a few players
query_players = list(profiles.keys())[:3]

for query in query_players:
    print(f"\n🔍 Similar players to: {query}")
    print("-" * 40)
    
    results = baseline.topk(query, k=5)
    
    for i, result in enumerate(results):
        role_match = "✓" if result['role_match'] else "✗"
        print(f"  {i+1}. {result['player_id']} | "
              f"Sim: {result['similarity']:.3f} | "
              f"Role: {result['role']} {role_match}")

## 5. Visualization

In [ ]:
# 5.1 Embedding space (PCA to 2D)
print("Visualizing embedding space...")

pca = PCA(n_components=2)
emb_2d = pca.fit_transform(baseline.embeddings)

fig, ax = plt.subplots(figsize=(12, 9))

# Color by role
unique_roles = list(set(r['primary_role'] for r in roles.values()))
role_to_color = {r: plt.cm.tab10(i) for i, r in enumerate(unique_roles)}

colors = [role_to_color[roles[pid]['primary_role']] for pid in player_ids]

scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=colors, alpha=0.7, s=60, edgecolors='black', linewidth=0.5)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=role_to_color[r], label=r) for r in unique_roles[:10]]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

ax.set_xlabel('PC1', fontsize=12)
ax.set_ylabel('PC2', fontsize=12)
ax.set_title('Baseline Embedding Space (PCA Projection)\nColored by Discovered Role', fontsize=14)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Similarity heatmap
print("Computing similarity matrix...")

subset = player_ids[:20]
sim_matrix = baseline.compute_similarity_matrix(subset)

fig, ax = plt.subplots(figsize=(14, 12))

sns.heatmap(
    sim_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0.5,
    ax=ax,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    annot_kws={"fontsize": 8}
)

ax.set_title('Player Similarity Matrix (Ratio-Based Baseline)', fontsize=14)
plt.tight_layout()
plt.show()

print("\n✅ Baseline evaluation complete!")